# Session 1 — Value iteration on FrozenLake

**Reinforcement Learning atelier, Université Lumière Lyon 2**

In this notebook you will:
1. explore a Gymnasium environment and see how it encodes an MDP $\langle \mathcal{S}, \mathcal{A}, \mathcal{P}, \mathcal{R}, \gamma \rangle$,
2. implement **value iteration** from the Bellman optimality equation,
3. extract an optimal policy and test it.

Levels: 🟢 **core** (everyone), 🟠 **stretch** (if you have time), 🔴 **bonus** (for the curious).

Each `TODO` is followed by a **check cell**: if it runs without error, you are on the right track.
Questions marked ❓ are for discussion with your partner; we go through them in the debrief.

In [ ]:
# Only if you work on Google Colab (gymnasium is not installed there): uncomment the next line
# %pip install -q gymnasium

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym

np.set_printoptions(precision=3, suppress=True)

---
## 🟢 Part 1 — The environment

FrozenLake is a $4 \times 4$ grid. The agent starts at `S` and must reach the goal `G` without falling into a hole `H`.

```
S F F F
F H F H
F F F H
H F F G
```

- Actions: `0` left, `1` down, `2` right, `3` up.
- Reward: `+1` when reaching `G`, `0` otherwise. The episode ends in `G` or in a hole.
- We start with `is_slippery=False`: the agent always moves where it wants.

In [ ]:
env = gym.make("FrozenLake-v1", map_name="4x4", is_slippery=False, render_mode="ansi")
state, info = env.reset(seed=0)
print("initial state:", state)
print(env.render())

print("number of states :", env.observation_space.n)
print("number of actions:", env.action_space.n)

### The Gymnasium interaction loop

Every RL agent talks to its environment through two functions:

- `state, info = env.reset()` starts a new episode,
- `next_state, reward, terminated, truncated, info = env.step(action)` takes one action.

`terminated` means the episode reached a terminal state of the MDP (goal or hole).
`truncated` means it was stopped by a time limit (100 steps for FrozenLake).

In [ ]:
# Take a few actions by hand: right, right, down
for action in [2, 2, 1]:
    next_state, reward, terminated, truncated, info = env.step(action)
    print(f"action={action} -> state={next_state}, reward={reward}, terminated={terminated}")
    print(env.render())

### TODO 1 — How good is a random agent?

Write a function that runs `n_episodes` episodes with **uniformly random actions** (`env.action_space.sample()`)
and returns the fraction of episodes that reach the goal.

In [ ]:
def run_random_agent(env, n_episodes=1000, seed=0):
    env.action_space.seed(seed)
    successes = 0
    for episode in range(n_episodes):
        state, info = env.reset(seed=seed + episode)
        done = False
        # TODO 1: loop until the episode ends, choosing random actions.
        #         Add 1 to `successes` when the goal is reached (reward == 1).
        raise NotImplementedError
    return successes / n_episodes

In [ ]:
# Check
success_random = run_random_agent(env, n_episodes=5000)
print(f"random agent success rate: {success_random:.3f}")
assert 0.0 < success_random < 0.05, "a random agent should almost never reach the goal (about 1%)"

### The model of the MDP: `env.unwrapped.P`

FrozenLake gives us its full model. `P[s][a]` is a list of tuples `(prob, next_state, reward, terminated)`:
all the possible outcomes of taking action `a` in state `s`.

This is exactly $\mathcal{P}^a_{ss'}$ and $\mathcal{R}^a_s$ from the lecture. **Value iteration needs it.**
From Session 2 on, our agents will *not* have access to `P`: they will only learn from `reset()` and `step()`.

In [ ]:
P = env.unwrapped.P
n_states = env.observation_space.n
n_actions = env.action_space.n

print("P[0][2]  (start, go right):", P[0][2])
print("P[14][2] (next to the goal, go right):", P[14][2])
print("P[5][0]  (state 5 is a hole):", P[5][0])

❓ **Questions**
1. In which states is `terminated` true? What happens to the agent in state 5?
2. Re-create the environment with `is_slippery=True` and print `P[0][1]`. Describe in words what "slippery" means.

---
## 🟢 Part 2 — Value iteration

**Bellman optimality equation** (lecture):

$$v_*(s) = \max_a \sum_{s'} \mathcal{P}^a_{ss'} \big( r + \gamma\, v_*(s') \big)$$

**Value iteration** turns this equation into an update rule, applied to all states at each sweep:

$$V_{k+1}(s) = \max_a \underbrace{\sum_{s'} \mathcal{P}^a_{ss'} \big( r + \gamma\, V_k(s') \big)}_{Q_k(s,a)}$$

We stop when $\max_s |V_{k+1}(s) - V_k(s)| < \theta$.

> **Terminal states.** If `terminated` is true, there is no future: the target is just `r`, not `r + gamma * V[s2]`.
> Multiply the future term by `(not terminated)`.

### TODO 2 — One-step lookahead

Compute $Q(s,a) = \sum_{s'} \mathcal{P}^a_{ss'} \big( r + \gamma V(s') \big)$ for **all** actions of one state `s`.

In [ ]:
def one_step_lookahead(P, V, s, gamma):
    # Returns a numpy array q of shape (n_actions,) with q[a] = Q(s, a)
    q = np.zeros(len(P[s]))
    # TODO 2: loop over actions a and over outcomes (prob, s2, r, terminated) in P[s][a]
    raise NotImplementedError
    return q

In [ ]:
# Check
V_test = np.zeros(n_states)
V_test[15] = 10.0
q14 = one_step_lookahead(P, V_test, 14, gamma=0.9)
assert np.allclose(q14, [0.0, 0.0, 1.0, 0.0]), q14            # only "right" reaches the goal; it is terminal
q10 = one_step_lookahead(P, V_test, 10, gamma=0.9)
assert np.allclose(q10, [0.0, 0.0, 0.0, 0.0]), q10            # no action from 10 reaches 15 in one step
V_test[14] = 4.0
q13 = one_step_lookahead(P, V_test, 13, gamma=0.5)
assert np.allclose(q13, [0.0, 0.0, 2.0, 0.0]), q13            # right from 13 -> 14 (not terminal): 0 + 0.5 * 4
print("TODO 2 OK")

### TODO 3 — Value iteration

Repeat sweeps over all states until convergence. Use **synchronous** updates: compute `V_new` from the old `V`.
Return the value function and the number of sweeps.

In [ ]:
def value_iteration(P, n_states, gamma=0.9, theta=1e-8, max_iters=10_000):
    V = np.zeros(n_states)
    for it in range(1, max_iters + 1):
        # TODO 3: compute V_new[s] = max_a Q(s, a) for every state, using one_step_lookahead
        #         stop (return V_new, it) when max |V_new - V| < theta
        raise NotImplementedError
        V = V_new
    return V, max_iters

In [ ]:
V_star, n_sweeps = value_iteration(P, n_states, gamma=0.9)
print(f"converged in {n_sweeps} sweeps")
print(V_star.reshape(4, 4))

# Check
assert np.isclose(V_star[0], 0.9 ** 5), "V(start) should be 0.9^5 = 0.59049"
assert np.isclose(V_star[14], 1.0)
print("TODO 3 OK")

❓ **Questions**
1. Why is $V(\text{start})$ exactly $0.9^5$? *(Hint: count the steps of the shortest path.)*
2. Why are the values of holes and of the goal equal to 0?
3. How many sweeps did it take? Relate this number to the length of the shortest path.

### TODO 4 — Extract the greedy policy

$$\pi_*(s) = \arg\max_a Q_*(s,a)$$

In [ ]:
def greedy_policy(P, V, n_states, gamma=0.9):
    # TODO 4: return an integer array of shape (n_states,) with the best action in each state
    raise NotImplementedError

In [ ]:
def run_policy(env, policy, n_episodes=1000, seed=0):
    # Runs a deterministic policy and returns the success rate
    successes = 0
    for episode in range(n_episodes):
        state, info = env.reset(seed=seed + episode)
        done = False
        while not done:
            state, reward, terminated, truncated, info = env.step(int(policy[state]))
            done = terminated or truncated
            successes += reward
    return successes / n_episodes

pi_star = greedy_policy(P, V_star, n_states, gamma=0.9)
success = run_policy(env, pi_star)
print(f"optimal policy success rate: {success:.3f}  (random agent: {success_random:.3f})")
assert success == 1.0
print("TODO 4 OK")

### Visualisation (provided)

In [ ]:
ARROWS = {0: "←", 1: "↓", 2: "→", 3: "↑"}

def plot_value_and_policy(env, V, policy, title=""):
    desc = env.unwrapped.desc.astype(str)
    nrow, ncol = desc.shape
    fig, ax = plt.subplots(figsize=(1.2 * ncol, 1.2 * nrow))
    im = ax.imshow(V.reshape(nrow, ncol), cmap="Blues")
    for s in range(nrow * ncol):
        r, c = divmod(s, ncol)
        cell = desc[r, c]
        label = cell if cell in "HG" else ARROWS[int(policy[s])]
        ax.text(c, r - 0.15, label, ha="center", va="center", fontsize=16)
        ax.text(c, r + 0.3, f"{V[s]:.2f}", ha="center", va="center", fontsize=8)
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(title)
    fig.colorbar(im, ax=ax, fraction=0.046)
    plt.show()

plot_value_and_policy(env, V_star, pi_star, "Deterministic FrozenLake, γ = 0.9")

---
## 🟠 Part 3 — Policy iteration (stretch)

Policy iteration alternates two steps (lecture, Session 2 slides):

1. **Policy evaluation**: compute $v_\pi$ for the current policy, by iterating the **Bellman expectation** equation
   $V(s) \leftarrow \sum_{s'} \mathcal{P}^{\pi(s)}_{ss'} \big( r + \gamma V(s') \big)$ until convergence.
2. **Policy improvement**: $\pi'(s) = \arg\max_a Q_\pi(s,a)$ (greedy with respect to $v_\pi$).

Stop when the policy does not change.

In [ ]:
def policy_evaluation(P, policy, n_states, gamma=0.9, theta=1e-8):
    V = np.zeros(n_states)
    while True:
        # TODO 5: one sweep of the Bellman expectation backup for the deterministic `policy`
        #         (only the action policy[s] is used), stop when the change is below theta
        raise NotImplementedError


def policy_iteration(P, n_states, n_actions, gamma=0.9):
    policy = np.zeros(n_states, dtype=int)  # start: always go left
    for it in range(1, 1000):
        # TODO 6: evaluate the policy, improve it greedily, stop when it is stable
        #         return (V, policy, it)
        raise NotImplementedError

In [ ]:
V_pi, pi_pi, n_improvements = policy_iteration(P, n_states, n_actions, gamma=0.9)
print(f"policy iteration: {n_improvements} improvement steps")

# Check: same optimal value function as value iteration
assert np.allclose(V_pi, V_star, atol=1e-6)
print("TODO 5-6 OK")

❓ **Questions**
1. Policy iteration needs only a few *improvement* steps. Is it cheaper than value iteration overall? (Each improvement step contains a full policy evaluation.)
2. Do `pi_pi` and `pi_star` always pick the same action? If not, why can both be optimal?

---
## 🔴 Part 4 — Slippery ice (bonus)

In [ ]:
env_slip = gym.make("FrozenLake-v1", map_name="4x4", is_slippery=True)
P_slip = env_slip.unwrapped.P

for gamma in [0.9, 0.99]:
    V_s, sweeps = value_iteration(P_slip, n_states, gamma=gamma)
    pi_s = greedy_policy(P_slip, V_s, n_states, gamma=gamma)
    rate = run_policy(env_slip, pi_s, n_episodes=2000)
    print(f"gamma={gamma}: {sweeps} sweeps, V(start)={V_s[0]:.3f}, success rate={rate:.3f}")
    plot_value_and_policy(env_slip, V_s, pi_s, f"Slippery FrozenLake, γ = {gamma}")

❓ **Questions**
1. Look at the action in state 0 and in state 6. Why does the optimal policy sometimes point *away* from the goal or *into a wall*?
2. With $\gamma = 0.9$, $V(\text{start}) \approx 0.07$, but the success rate is $\approx 73\%$. Is there a contradiction? What does $V$ measure here?
3. The number of sweeps is about 4 times larger for $\gamma = 0.99$. The contraction bound from the slides predicts a factor of about 10. Why is the bound pessimistic?
4. Try `map_name="8x8"`. What changes?

---
## Wrap-up

- Value iteration = repeatedly applying the **Bellman optimality backup**. It converges because the backup is a $\gamma$-contraction.
- It needs the **model** `P`. In most real problems we don't have it.

**Session 2:** learning *without* a model, only from `reset()` and `step()`: Monte Carlo, TD learning, SARSA and Q-learning.